In [ ]:
import os
from pathlib import Path
import json
import pickle

import numpy as np
from sklearn.decomposition import PCA

# Keep paths local by default. Override this when embeddings are stored elsewhere.
RESULT_DIR = Path(os.environ.get("LLM_EMB_RESULT_DIR", "."))
# print(RESULT_DIR)
INPUT_JSONLINE_FILE = RESULT_DIR / "item_input_avg_emb.jsonline"
GEN_JSONLINE_FILE = RESULT_DIR / "item_gen_avg_emb.jsonline"
INPUT_PKL_FILE = RESULT_DIR / "item_input_avg_emb.pkl"
GEN_PKL_FILE = RESULT_DIR / "item_gen_avg_emb.pkl"
CONCAT_OUTPUT_FILE = RESULT_DIR / "item_pca384_concat768.pkl"

N_COMPONENTS_EACH = 384
RANDOM_STATE = 42


def read_jsonline(file_path):
    """Read valid JSON objects from a jsonline file."""
    records = []
    skipped = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                skipped += 1

    if skipped:
        print(f"Skipped {skipped} invalid JSON lines in {file_path}")
    return records


def records_to_embedding_matrix(records):
    """Convert hidden_states records to a 2D matrix ordered by item_id."""
    item_to_hidden = {}
    for record in records:
        if "item_id" not in record or "hidden_states" not in record:
            continue
        item_to_hidden[int(record["item_id"])] = record["hidden_states"]

    expected_ids = range(1, len(item_to_hidden) + 1)
    missing_ids = [item_id for item_id in expected_ids if item_id not in item_to_hidden]
    if missing_ids:
        raise ValueError(f"Missing item_id values, first missing ids: {missing_ids[:10]}")

    embeddings = np.asarray([item_to_hidden[item_id] for item_id in expected_ids], dtype=np.float32)
    while embeddings.ndim > 2 and embeddings.shape[1] == 1:
        embeddings = np.squeeze(embeddings, axis=1)

    if embeddings.ndim != 2:
        raise ValueError(f"Expected a 2D embedding matrix, got shape {embeddings.shape}")
    return embeddings


def jsonline_to_embeddings(jsonline_file, output_pkl_file):
    records = read_jsonline(jsonline_file)
    embeddings = records_to_embedding_matrix(records)

    with open(output_pkl_file, "wb") as f:
        pickle.dump(embeddings, f)

    print(f"Loaded {len(records)} records from: {jsonline_file}")
    print(f"Embedding shape: {embeddings.shape}")
    print(f"Saved raw embeddings to: {output_pkl_file}")
    return embeddings


def pca_reduce(embeddings, n_components=384, random_state=42):
    pca = PCA(n_components=n_components, random_state=random_state)
    reduced_embeddings = pca.fit_transform(embeddings)
    explained_variance = float(np.sum(pca.explained_variance_ratio_))
    return reduced_embeddings, pca, explained_variance


def pca_then_concat(embeddings1, embeddings2, output_file, n_components_each=384, random_state=42):
    print(f"Embedding 1 shape: {embeddings1.shape}")
    print(f"Embedding 2 shape: {embeddings2.shape}")

    if embeddings1.shape[0] != embeddings2.shape[0]:
        raise ValueError(f"Sample counts do not match: {embeddings1.shape[0]} vs {embeddings2.shape[0]}")

    reduced1, pca1, variance1 = pca_reduce(embeddings1, n_components_each, random_state)
    reduced2, pca2, variance2 = pca_reduce(embeddings2, n_components_each, random_state)
    concat_embeddings = np.concatenate([reduced1, reduced2], axis=1)

    with open(output_file, "wb") as f:
        pickle.dump(concat_embeddings, f)

    print(f"Reduced embedding 1 shape: {reduced1.shape}, variance: {variance1:.4%}")
    print(f"Reduced embedding 2 shape: {reduced2.shape}, variance: {variance2:.4%}")
    print(f"Concat embedding shape: {concat_embeddings.shape}")
    print(f"Saved concat embeddings to: {output_file}")

    return concat_embeddings, {
        "pca1": pca1,
        "pca2": pca2,
        "variance1": variance1,
        "variance2": variance2,
        "reduced1": reduced1,
        "reduced2": reduced2,
    }


input_embeddings = jsonline_to_embeddings(INPUT_JSONLINE_FILE, INPUT_PKL_FILE)
gen_embeddings = jsonline_to_embeddings(GEN_JSONLINE_FILE, GEN_PKL_FILE)

concat_embeddings, pca_info = pca_then_concat(
    input_embeddings,
    gen_embeddings,
    CONCAT_OUTPUT_FILE,
    n_components_each=N_COMPONENTS_EACH,
    random_state=RANDOM_STATE,
)


In [ ]:
import inspect

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.manifold import TSNE

# Visualize the input/gen embeddings loaded in the previous cell.
MAX_POINTS = 1000
PERPLEXITY = 30
N_ITER = 1000


def sample_embeddings(embeddings, max_points=None):
    if max_points is None:
        return embeddings
    return embeddings[:max_points]


def build_tsne(perplexity=30, n_iter=1000, random_state=42):
    tsne_kwargs = dict(
        n_components=2,
        perplexity=perplexity,
        random_state=random_state,
        init="pca",
        learning_rate="auto",
    )
    iter_arg = "max_iter" if "max_iter" in inspect.signature(TSNE).parameters else "n_iter"
    tsne_kwargs[iter_arg] = n_iter
    return TSNE(**tsne_kwargs)


def tsne_project_two_embeddings(embeddings1, embeddings2, perplexity=30, n_iter=1000, random_state=42):
    if embeddings1.shape[1] == embeddings2.shape[1]:
        combined = np.vstack([embeddings1, embeddings2])
        projected = build_tsne(perplexity, n_iter, random_state).fit_transform(combined)
        return projected[: len(embeddings1)], projected[len(embeddings1) :]

    projected1 = build_tsne(perplexity, n_iter, random_state).fit_transform(embeddings1)
    projected2 = build_tsne(perplexity, n_iter, random_state).fit_transform(embeddings2)
    return projected1, projected2


def visualize_two_embeddings_without_lines(
    embeddings1,
    embeddings2,
    label1="Input avg embedding",
    label2="Generation avg embedding",
    max_points=1000,
    perplexity=30,
    n_iter=1000,
    random_state=42,
):
    embeddings1 = sample_embeddings(embeddings1, max_points)
    embeddings2 = sample_embeddings(embeddings2, max_points)

    if embeddings1.shape[0] != embeddings2.shape[0]:
        raise ValueError(f"Sample counts do not match: {embeddings1.shape[0]} vs {embeddings2.shape[0]}")

    print(f"{label1} shape: {embeddings1.shape}")
    print(f"{label2} shape: {embeddings2.shape}")
    print("Running t-SNE...")

    projected1, projected2 = tsne_project_two_embeddings(
        embeddings1,
        embeddings2,
        perplexity=perplexity,
        n_iter=n_iter,
        random_state=random_state,
    )

    colors1 = plt.cm.Blues(np.linspace(0.35, 1.0, len(projected1)))
    colors2 = plt.cm.Reds(np.linspace(0.35, 1.0, len(projected2)))

    plt.figure(figsize=(12, 10))
    plt.scatter(projected1[:, 0], projected1[:, 1], c=colors1, alpha=0.75, s=35, edgecolors="none")
    plt.scatter(projected2[:, 0], projected2[:, 1], c=colors2, alpha=0.75, s=35, edgecolors="none")

    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=colors1[-1], markersize=10, label=label1),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=colors2[-1], markersize=10, label=label2),
    ]
    plt.legend(handles=legend_elements, loc="best")
    plt.title(f"t-SNE visualization without connection lines (perplexity={perplexity}, n_iter={n_iter})")
    plt.grid(True, linestyle="--", alpha=0.35)
    plt.tight_layout()
    plt.show()

    return projected1, projected2


projected_input, projected_gen = visualize_two_embeddings_without_lines(
    input_embeddings,
    gen_embeddings,
    max_points=MAX_POINTS,
    perplexity=PERPLEXITY,
    n_iter=N_ITER,
    random_state=RANDOM_STATE,
)
